In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Spread-Skill plots

In [ ]:
def bin_and_average_flat(self, spread_flat, rmse_flat, bins):
    """
    Bin the pixel-wise spread and compute the average spread and RMSE for each bin.
    """
    bin_indices = np.digitize(spread_flat, bins)
    mean_spread = []
    mean_rmse = []

    for b in range(1, len(bins)):
        bin_mask = bin_indices == b
        if np.any(bin_mask):  # Ensure the bin is not empty
            mean_spread.append(np.mean(spread_flat[bin_mask]))
            mean_rmse.append(np.mean(rmse_flat[bin_mask]))

    return mean_spread, mean_rmse

In [ ]:
# Prediction and target: (pred_steps, Nlat, Nlon, features)
# Iterate over variables
paths_dict = defaultdict(list)
units = [obj.dataset_info.units[name] for name in feature_names]
var_figs = []
LT_seq = [10,20,44]
purple_shades = ["#4b0082", "#9933cc", "#e0b3ff"]  # Light, Medium, Dark purple

for var_i, (var_name, var_unit) in enumerate(
        zip(feature_names, units)
    ):

    plt.figure(figsize=(10, 6))

    for curve in range(len(LT_seq)):

        # Flatten arrays to handle all pixels across all images
        spread_flat = prediction_std[:LT_seq[curve],..., var_i].cpu().numpy().flatten()
        rmse_flat = rmse[:LT_seq[curve],..., var_i].cpu().numpy().flatten()

        # Define bins for spread
        num_bins = 20
        bins = np.linspace(spread_flat.min(), 
                        spread_flat.max(), 
                        num_bins + 1)
        
        mean_spread, mean_rmse = self.bin_and_average_flat(spread_flat, rmse_flat, bins)

        # Plot spread-RMSE relationship for both models
        plt.plot(mean_spread, mean_rmse, marker="x", label=var_name+" LT 0-"+str(LT_seq[curve]), color=purple_shades[curve])

    # Add perfect line y=x
    max_value = max(np.max(mean_spread),np.max(mean_spread),np.max(mean_rmse),
                    np.max(mean_rmse))

    plt.plot([0, max_value], [0, max_value], linestyle="--", color="gray", label="Perfect")

    plt.xlabel("Spread (Pixel-wise Standard Deviation across Ensemble)")
    plt.ylabel("Skill (Pixel-wise Root Mean Squared Error)")
    plt.title(var_name+" Spread-Skill Plot")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

## Ratio Spread-Error

## CRPS curves